# HW4 Part 1 — Colab environment and Q1 runner

Keep this notebook in your local repository. Select **Colab → New Colab Server → GPU (T4 or L4)** in VS Code and sign in. A local Python kernel will not work.

First right-click your local **part1** folder in VS Code Explorer and choose **Upload to Colab**. The expected remote location is `/content/part1`. Local edits do not automatically update this uploaded copy: save and upload again after every edit. This notebook never fills in homework solutions.

Run cells individually, in order. Do not use Run All. The Python 3.11 environment below is separate from Colab's notebook Python and from the later Part 2 environment. Re-run setup after Colab deletes/replaces the runtime.


In [3]:
from pathlib import Path
import subprocess
import sys

assert sys.platform == "linux" and Path("/content").exists(), "Select a Colab GPU kernel, not local Python."
subprocess.run(["nvidia-smi"], check=True)
PART1 = Path("/content/part1")
assert (PART1 / "main.py").is_file(), "Upload your local part1 folder to Colab first; expected /content/part1/main.py."
assert (PART1 / "requirements.txt").is_file()
print("Remote code:", PART1)
print((PART1 / "requirements.txt").read_text())


Remote code: /content/part1
transformers==4.26.1
datasets==2.19.2
torch==2.1.2
evaluate==0.4.0
scikit-learn==1.2.1
nltk==3.8.1
numpy<2
pyarrow<21
pandas<2.2
fsspec<2023.9



## Install the professor's Part 1 dependencies

This downloads a separate Python 3.11 interpreter and the pinned packages, including GPU-enabled PyTorch. It can take several minutes and uses several GB of runtime disk space. The notebook kernel stays on Colab's existing Python; all homework commands explicitly use the new environment.


In [4]:
subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
UV = [sys.executable, "-m", "uv"]
subprocess.run(UV + ["python", "install", "3.11"], check=True)
ENV = Path("/content/hw4-part1-env")
if not (ENV / "bin/python").exists():
    subprocess.run(UV + ["venv", "--python", "3.11", "--seed", str(ENV)], check=True)
PYTHON = str(ENV / "bin/python")
subprocess.run(UV + ["pip", "install", "--python", PYTHON, "-r", str(PART1 / "requirements.txt")], check=True)
subprocess.run(UV + ["pip", "check", "--python", PYTHON], check=True)
subprocess.run([PYTHON, "-c", "import nltk; nltk.download('wordnet', raise_on_error=True); nltk.download('punkt', raise_on_error=True)"], check=True)


CompletedProcess(args=['/content/hw4-part1-env/bin/python', '-c', "import nltk; nltk.download('wordnet', raise_on_error=True); nltk.download('punkt', raise_on_error=True)"], returncode=0)

In [5]:
check = """
import sys, torch, transformers, datasets, evaluate, sklearn, nltk, numpy, pyarrow, pandas, fsspec
print('Python:', sys.version)
for module in [torch, transformers, datasets, evaluate, sklearn, nltk, numpy, pyarrow, pandas, fsspec]:
    print(module.__name__ + ':', module.__version__)
assert sys.version_info[:2] == (3, 11)
assert torch.cuda.is_available(), 'No GPU available. Reconnect to a Colab GPU runtime.'
x = torch.ones(4, device='cuda')
assert (x + x).sum().item() == 8
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
print('ENVIRONMENT READY')
"""
subprocess.run([PYTHON, "-c", check], check=True)


CompletedProcess(args=['/content/hw4-part1-env/bin/python', '-c', "\nimport sys, torch, transformers, datasets, evaluate, sklearn, nltk, numpy, pyarrow, pandas, fsspec\nprint('Python:', sys.version)\nfor module in [torch, transformers, datasets, evaluate, sklearn, nltk, numpy, pyarrow, pandas, fsspec]:\n    print(module.__name__ + ':', module.__version__)\nassert sys.version_info[:2] == (3, 11)\nassert torch.cuda.is_available(), 'No GPU available. Reconnect to a Colab GPU runtime.'\nx = torch.ones(4, device='cuda')\nassert (x + x).sum().item() == 8\nprint('GPU:', torch.cuda.get_device_name(0))\nprint('GPU memory (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))\nprint('ENVIRONMENT READY')\n"], returncode=0)

In [6]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")

saved = Path(
    "/content/drive/MyDrive/NYU_NLP_HW4/part1/"
    "20261109T004132335321Z"
)

assert (saved / "out/config.json").exists(), "Check the backup path."
shutil.copytree(saved / "out", PART1 / "out", dirs_exist_ok=True)
shutil.copy2(saved / "out_original.txt", PART1 / "out_original.txt")

print("Q1 model restored. Ready for Q2.")

Mounted at /content/drive
Q1 model restored. Ready for Q2.


## Q1 debug run — only after entering your Q1 solution

Save `part1/main.py` locally, then upload the **part1** folder again. This cell checks that `do_train` in the remote copy no longer contains `NotImplementedError`. Q2/Q3 placeholders should remain. The debug target is accuracy above 0.88; report the actual result before proceeding to full training. The first run also downloads BERT and IMDB.


In [5]:
import ast
source = (PART1 / "main.py").read_text()
tree = ast.parse(source)
fn = next(n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name == "do_train")
assert not any(isinstance(n, ast.Name) and n.id == "NotImplementedError" for n in ast.walk(fn)), "Enter Q1 locally, save, and upload part1 again."
print(ast.get_source_segment(source, fn))
subprocess.run([PYTHON, "main.py", "--train", "--eval", "--debug_train"], cwd=PART1, check=True)


def do_train(args, model, train_dataloader, save_dir="./out"):
    optimizer = AdamW(model.parameters(), lr=args.learning_rate)
    num_epochs = args.num_epochs
    num_training_steps = num_epochs * len(train_dataloader)
    lr_scheduler = get_scheduler(
        name="linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps
    )
    model.train()
    progress_bar = tqdm(range(num_training_steps))

    ################################
    ##### YOUR CODE BEGINGS HERE ###

    # Implement the training loop --- make sure to use the optimizer and lr_sceduler (learning rate scheduler)
    # Remember that pytorch uses gradient accumumlation so you need to use zero_grad (https://pytorch.org/tutorials/recipes/recipes/zeroing_out_gradients.html)
    # You can use progress_bar.update(1) to see the progress during training
    # You can refer to the pytorch tutorial covered in class for reference

    for epoch in range(num_epochs):
        for batch in train_datal

CompletedProcess(args=['/content/hw4-part1-env/bin/python', 'main.py', '--train', '--eval', '--debug_train'], returncode=0)

In [6]:
from pathlib import Path

values = [
    int(x)
    for x in Path("/content/part1/out_original.txt").read_text().split()
]
predictions = values[0::2]
labels = values[1::2]

assert len(predictions) == len(labels) == 1000
accuracy = sum(p == y for p, y in zip(predictions, labels)) / len(labels)
print(f"Debug accuracy: {accuracy:.2%}")

Debug accuracy: 88.30%


## Full Q1 run — wait until the debug result has been checked

Set `RUN_FULL = True` only when ready. Full training replaces the debug checkpoint in `out/` and predictions in `out_original.txt`. The full-test target is accuracy above 0.91. GPU runtime duration can differ from the handout's estimate.


In [7]:
RUN_FULL = True
if RUN_FULL:
    subprocess.run([PYTHON, "main.py", "--train", "--eval"], cwd=PART1, check=True)
else:
    print("Skipped. Complete and check the debug run first.")


In [11]:
from pathlib import Path

values = [
    int(x)
    for x in Path("/content/part1/out_original.txt").read_text().split()
]
predictions = values[0::2]
labels = values[1::2]

assert len(predictions) == len(labels) == 25000
accuracy = sum(p == y for p, y in zip(predictions, labels)) / len(labels)
print(f"Full-test accuracy: {accuracy:.2%}")

Full-test accuracy: 92.81%


## Back up a completed run to Google Drive

Mount Drive when prompted; if the extension cannot show the authorization flow, open this same notebook in browser Colab to mount and back up there. Each backup has a new timestamp and will not replace an earlier backup. This saves the uploaded source, predictions, and final model checkpoint, but not the environment or dataset cache. `out/` is the checkpoint, not an optimizer-state training resume file. Back up before disconnecting. Only predictions from a verified full run should be submitted for Q1.


In [10]:
from google.colab import drive
from datetime import datetime, timezone
import shutil

drive.mount("/content/drive")
backup = Path("/content/drive/MyDrive/NYU_NLP_HW4/part1") / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
backup.mkdir(parents=True, exist_ok=False)
for name in ["main.py", "utils.py", "requirements.txt", "out_original.txt"]:
    src = PART1 / name
    if src.exists():
        shutil.copy2(src, backup / name)
if (PART1 / "out").exists():
    shutil.copytree(PART1 / "out", backup / "out")
print("Backup saved:", backup)


Mounted at /content/drive
Backup saved: /content/drive/MyDrive/NYU_NLP_HW4/part1/20261109T004132335321Z


In [6]:
import subprocess

def run_part1(*arguments):
    with subprocess.Popen(
        [PYTHON, "-u", "main.py", *arguments],
        cwd=PART1,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)

        if process.wait() != 0:
            raise RuntimeError("Run failed; check the output above.")

run_part1("--eval_transformed", "--debug_transformation")

/content/hw4-part1-env/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(





Generating train split: 100%|██████████| 25000/25000 [00:00<00:00, 171213.69 examples/s]

Generating test split: 100%|██████████| 25000/25000 [00:00<00:00, 190593.00 examples/s]

Generating unsupervised split: 100%|██████████| 50000/50000 [00:00<00:00, 188901.64 examples/s]

Map: 100%|██████████| 25000/25000 [00:10<00:00, 2464.32 examples/s]

Map: 100%|██████████| 25000/25000 [00:09<00:00, 2522.57 examples/s]

Map: 100%|██████████| 50000/50000 [00:20<00:00, 2496.04 examples/s]
Actual training...
len(train_dataloader): 3125
len(eval_dataloader): 3125

Map: 100%|██████████| 5/5 [00:00<00:00, 1031.71 examples/s]
Original Example  0
{'text': "<br /><br />When I unsuspectedly rented A Thousand Ac

In [7]:
run_part1("--eval_transformed")

/content/hw4-part1-env/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(

Map: 100%|██████████| 25000/25000 [00:10<00:00, 2442.74 examples/s]
Actual training...
len(train_dataloader): 3125
len(eval_dataloader): 3125

Map: 100%|██████████| 25000/25000 [00:06<00:00, 3619.07 examples/s]

Map: 100%|██████████| 25000/25000 [00:10<00:00, 2397.27 examples/s]


100%|██████████| 3125/3125 [03:11<00:00, 16.33it/s]
Score:  {'accuracy': 0.88668}


In [8]:
from pathlib import Path

values = [
    int(x)
    for x in (PART1 / "out_transformed.txt").read_text().split()
]
predictions = values[0::2]
labels = values[1::2]

assert len(predictions) == len(labels) == 25000
accuracy = sum(p == y for p, y in zip(predictions, labels)) / len(labels)

print(f"Transformed-test accuracy: {accuracy:.2%}")
print(f"Drop from Q1: {92.81 - 100 * accuracy:.2f} percentage points")

Transformed-test accuracy: 88.67%
Drop from Q1: 4.14 percentage points


In [9]:
from datetime import datetime, timezone
import shutil

q2_backup = (
    Path("/content/drive/MyDrive/NYU_NLP_HW4/part1")
    / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
)
q2_backup.mkdir(parents=True, exist_ok=False)

for name in [
    "main.py",
    "utils.py",
    "requirements.txt",
    "out_original.txt",
    "out_transformed.txt",
]:
    src = PART1 / name
    if src.exists():
        shutil.copy2(src, q2_backup / name)

if (PART1 / "out").exists():
    shutil.copytree(PART1 / "out", q2_backup / "out")

print("Q2 backup saved:", q2_backup)

Q2 backup saved: /content/drive/MyDrive/NYU_NLP_HW4/part1/20261109T035332740693Z


In [7]:
q3_check = """
from types import SimpleNamespace
import torch
import main
from datasets import load_dataset
from transformers import AutoTokenizer

assert torch.cuda.is_available(), "Connect to a Colab GPU first."

main.tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
dataset = load_dataset("imdb")
loader = main.create_augmented_dataloader(
    SimpleNamespace(batch_size=8),
    dataset,
)

assert len(loader.dataset) == 30000
assert len(loader) == 3750

batch = next(iter(loader))
assert set(batch) == {"input_ids", "attention_mask", "token_type_ids", "labels"}
assert batch["input_ids"].shape == (8, 512)
assert batch["labels"].shape == (8,)
assert set(batch["labels"].tolist()) <= {0, 1}

print("Q3 DATALOADER CHECK PASSED")
"""

import subprocess

print("Starting Q3 check...", flush=True)

with subprocess.Popen(
    [PYTHON, "-u", "-c", q3_check],
    cwd=PART1,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
) as process:
    for line in process.stdout:
        print(line, end="", flush=True)

    returncode = process.wait()

if returncode != 0:
    raise RuntimeError("Q3 check failed; see the output above.")

print("Check finished successfully.")

Starting Q3 check...
/content/hw4-part1-env/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(





Generating train split: 100%|██████████| 25000/25000 [00:00<00:00, 133477.26 examples/s]

Generating test split: 100%|██████████| 25000/25000 [00:00<00:00, 165259.16 examples/s]

Generating unsupervised split: 100%|██████████| 50000/50000 [00:00<00:00, 135529.39 examples/s]

Map: 100%|██████████| 5000/5000 [00:01<00:00, 3036.67 examples/s]

Map: 100%|██████████| 30000/30000 [00:30<00:00, 983.87 examples/s] 
Augmented training examples: 30000
Augmented training batches: 3750
Q3 DATALOADER CHECK PASSED
Check finished successfully.
